# Exercise: Implementing a PagedAttention Simulator

Welcome! In this exercise, you will implement a simplified key-value (KV) cache manager that mimics the core logic of PagedAttention. This will give you a hands-on understanding of how vLLM efficiently manages memory for large language models.

By the end of this notebook, you will be able to:
- Implement a system to allocate fixed-size memory blocks for different sequences.
- Write logic to append new tokens to a sequence, allocating more memory only when needed.
- Map a logical token position within a sequence to its physical location in a memory block.

Let's get started!

In [ ]:
import dataclasses
from typing import List, Dict, Tuple, Optional

# Helper Class - You do not need to modify this
@dataclasses.dataclass
class KVBlock:
    """Represents a single, fixed-size block of physical memory for the KV cache."""
    block_num: int

# Main Class - You will be implementing methods for this class
class SimulatedKVCacheManager:
    """
    Manages the allocation of KVBlocks for different sequences, simulating PagedAttention.
    """
    def __init__(self, block_size: int, num_blocks: int):
        """
        Initializes the KVCacheManager.

        Args:
            block_size: The number of tokens each KVBlock can hold.
            num_blocks: The total number of physical KVBlocks available.
        """
        self.block_size = block_size
        self.num_blocks = num_blocks
        
        # A list of integers representing free block numbers.
        # We'll treat this as a stack, popping from the end to get a free block.
        self.free_blocks: List[int] = list(range(self.num_blocks))
        
        # Maps a sequence ID to the list of KVBlocks allocated to it.
        # This is the "block table" for each sequence.
        self.block_tables: Dict[int, List[KVBlock]] = {}
        
        # Maps a sequence ID to its current token length.
        self.sequence_lengths: Dict[int, int] = {}
        
    def _allocate_block(self) -> Optional[KVBlock]:
        """
        (Private helper method)
        Allocates a single free block from the pool.
        
        Returns:
            A KVBlock if one is available, otherwise None.
        """
        if not self.free_blocks:
            return None
        block_num = self.free_blocks.pop()
        return KVBlock(block_num=block_num)

    # You will implement the following methods in the exercises below.
    def allocate_sequence(self, seq_id: int) -> bool:
        raise NotImplementedError("You must implement this method in Exercise 1.")
        
    def append_token(self, seq_id: int) -> bool:
        raise NotImplementedError("You must implement this method in Exercise 2.")
        
    def get_physical_location(self, seq_id: int, token_idx: int) -> Optional[Tuple[int, int]]:
        raise NotImplementedError("You must implement this method in Exercise 3.")

### Exercise 1: Allocating a New Sequence

Your first task is to implement the `allocate_sequence` method. When a new request (sequence) arrives, the manager needs to allocate its *first* memory block.

**Instructions:**
1.  Check if the `seq_id` has already been allocated. If so, you cannot allocate it again, so you should return `False`.
2.  Use the provided helper method `self._allocate_block()` to get a new block from the free pool.
3.  If a block is successfully allocated:
    -   Create a new entry in `self.block_tables` for the `seq_id`. This entry should be a list containing the newly allocated block.
    -   Initialize the sequence length for this `seq_id` to 1 in `self.sequence_lengths`.
    -   Return `True`.
4.  If `_allocate_block()` returns `None` (meaning no blocks are free), you cannot allocate the sequence. Return `False`.

**Hint:** A Python dictionary's `in` operator is a great way to check if a `seq_id` already exists as a key.

In [ ]:
def allocate_sequence(self, seq_id: int) -> bool:
    """
    Allocates the initial block for a new sequence.

    Args:
        seq_id: The unique identifier for the new sequence.

    Returns:
        True if allocation was successful, False otherwise.
    """
    # Prevent re-allocation of an existing sequence
    if seq_id in self.block_tables:
        return False
    
    ### START CODE HERE ### (≈ 6 lines)
    # Attempt to allocate a block from the free pool.
    new_block = self._allocate_block()
    
    # If a block is available, initialize the sequence's state.
    if new_block:
        self.block_tables[seq_id] = [new_block]
        self.sequence_lengths[seq_id] = 1
        return True
    else:
        # No free blocks were available.
        return False
    ### END CODE HERE ###

# Monkey-patch the method into our class for testing
SimulatedKVCacheManager.allocate_sequence = allocate_sequence

In [ ]:
# Test cell for Exercise 1

print("--- Testing Exercise 1: allocate_sequence ---")

# Test case 1: Successful allocation
manager = SimulatedKVCacheManager(block_size=4, num_blocks=2)
result1 = manager.allocate_sequence(seq_id=101)
print(f"Allocating sequence 101... Success: {result1}")
assert result1 is True, "Expected allocation to succeed, but it failed."
assert 101 in manager.block_tables, "seq_id 101 not found in block_tables."
assert len(manager.block_tables[101]) == 1, f"Expected 1 block for seq 101, but got {len(manager.block_tables[101])}."
assert manager.sequence_lengths[101] == 1, f"Expected length 1 for seq 101, but got {manager.sequence_lengths[101]}."
assert len(manager.free_blocks) == 1, f"Expected 1 free block remaining, but got {len(manager.free_blocks)}."

# Test case 2: Attempt to allocate an existing sequence
result2 = manager.allocate_sequence(seq_id=101)
print(f"Re-allocating sequence 101... Success: {result2}")
assert result2 is False, "Expected re-allocation to fail, but it succeeded."

# Test case 3: Allocation when no blocks are free
manager.allocate_sequence(seq_id=102) # Use the last block
result3 = manager.allocate_sequence(seq_id=103)
print(f"Allocating sequence 103 with no free blocks... Success: {result3}")
assert result3 is False, "Expected allocation to fail when no blocks are free."
assert len(manager.free_blocks) == 0, f"Expected 0 free blocks, but got {len(manager.free_blocks)}."

print("\n✅ All tests for Exercise 1 passed!")

### Exercise 2: Appending a Token to a Sequence

Great job! Now that you can start a sequence, you need to handle the generation of subsequent tokens. The `append_token` method simulates adding one new token to an existing sequence.

The key idea of PagedAttention is that a new physical block is only allocated when the current one runs out of space.

**Instructions:**
1.  Check if the `seq_id` exists. If not, you can't append to it, so return `False`.
2.  Get the current length of the sequence from `self.sequence_lengths`.
3.  Check if the last block is full. This happens when the sequence length is a multiple of `self.block_size`.
    -   If it is full, you must call `self._allocate_block()` to get a new block.
    -   If a new block is available, append it to the list of blocks for this sequence in `self.block_tables`.
    -   If no new block is available (`_allocate_block()` returns `None`), you're out of memory. Return `False`.
4.  If the last block was not full, or if you successfully allocated a new one, increment the sequence's length in `self.sequence_lengths`.
5.  Return `True` to indicate success.

In [ ]:
def append_token(self, seq_id: int) -> bool:
    """
    Appends a new token to an existing sequence, allocating a new block if necessary.

    Args:
        seq_id: The ID of the sequence to append to.

    Returns:
        True if the token was appended successfully, False otherwise (e.g., out of memory).
    """
    # Sequence must already exist to append to it.
    if seq_id not in self.block_tables:
        return False

    ### START CODE HERE ### (≈ 9 lines)
    current_len = self.sequence_lengths[seq_id]
    
    # Check if the last block is full. A new block is needed if the
    # number of tokens is perfectly divisible by the block size.
    if current_len % self.block_size == 0:
        new_block = self._allocate_block()
        if new_block:
            self.block_tables[seq_id].append(new_block)
        else:
            # Out of memory, cannot append token.
            return False
            
    # Increment the sequence length.
    self.sequence_lengths[seq_id] += 1
    return True
    ### END CODE HERE ###

# Monkey-patch the method into our class for testing
SimulatedKVCacheManager.append_token = append_token

In [ ]:
# Test cell for Exercise 2

print("--- Testing Exercise 2: append_token ---")

manager = SimulatedKVCacheManager(block_size=2, num_blocks=3)

# Setup: Allocate a sequence
manager.allocate_sequence(seq_id=201)
print(f"Initial state: seq_len={manager.sequence_lengths[201]}, num_blocks={len(manager.block_tables[201])}")

# Test case 1: Append within the first block
result1 = manager.append_token(seq_id=201) # len -> 2
print(f"Appending token 2... Success: {result1}, seq_len={manager.sequence_lengths[201]}, num_blocks={len(manager.block_tables[201])}")
assert result1 is True, "Append should succeed."
assert manager.sequence_lengths[201] == 2, f"Expected length 2, got {manager.sequence_lengths[201]}."
assert len(manager.block_tables[201]) == 1, "Should not have allocated a new block yet."

# Test case 2: Append when the block is full, requiring a new block
result2 = manager.append_token(seq_id=201) # len -> 3
print(f"Appending token 3... Success: {result2}, seq_len={manager.sequence_lengths[201]}, num_blocks={len(manager.block_tables[201])}")
assert result2 is True, "Append that triggers new block allocation should succeed."
assert manager.sequence_lengths[201] == 3, f"Expected length 3, got {manager.sequence_lengths[201]}."
assert len(manager.block_tables[201]) == 2, "Should have allocated a second block."

# Test case 3: Run out of memory
manager.append_token(seq_id=201) # len -> 4
manager.append_token(seq_id=201) # len -> 5, allocates 3rd and final block
print(f"State before OOM: seq_len={manager.sequence_lengths[201]}, num_blocks={len(manager.block_tables[201])}")
assert len(manager.free_blocks) == 0, "All blocks should be used now."

result3 = manager.append_token(seq_id=201) # len -> 6, needs a 4th block which doesn't exist
print(f"Appending token 6 (expect OOM)... Success: {result3}")
assert result3 is False, "Append should fail when out of memory."
assert manager.sequence_lengths[201] == 5, "Sequence length should not increase on failure."

# Test case 4: Append to a non-existent sequence
result4 = manager.append_token(seq_id=999)
print(f"Appending to non-existent seq 999... Success: {result4}")
assert result4 is False, "Should not be able to append to a sequence that doesn't exist."

print("\n✅ All tests for Exercise 2 passed!")

### Exercise 3: Finding a Token's Physical Location

Excellent! You've built the core memory management logic. The final piece of the puzzle is mapping a *logical* position to a *physical* one.

A logical position is defined by `(seq_id, token_idx)`, where `token_idx` is the 0-indexed position of the token within its sequence. A physical location is `(block_num, offset)`, where `block_num` is the global, unique number of the physical block and `offset` is the position inside that block.

**Instructions:**
1.  Check that the `seq_id` exists and that the `token_idx` is valid (i.e., it's not out of bounds for the sequence's current length). If not, return `None`.
2.  Calculate which block in the sequence's block list contains the token. For example, in a system with `block_size=4`, `token_idx=7` would be in the 2nd block (index 1) of that sequence's list.
3.  Calculate the offset of the token within that block. For `token_idx=7` and `block_size=4`, the offset would be 3.
4.  Use the calculated block index to retrieve the correct `KVBlock` object from `self.block_tables`.
5.  Return a tuple containing the physical block's number (`KVBlock.block_num`) and the calculated offset.

**Hint:** The integer division `//` and modulo `%` operators are your best friends for this task!

In [ ]:
def get_physical_location(self, seq_id: int, token_idx: int) -> Optional[Tuple[int, int]]:
    """
    Maps a logical token index in a sequence to its physical (block_num, offset) pair.

    Args:
        seq_id: The ID of the sequence.
        token_idx: The 0-indexed position of the token within the sequence.

    Returns:
        A tuple (block_num, offset) if the location is valid, otherwise None.
    """
    # Check for invalid sequence ID or out-of-bounds token index.
    if seq_id not in self.sequence_lengths or token_idx >= self.sequence_lengths[seq_id]:
        return None
        
    ### START CODE HERE ### (≈ 5 lines)
    # Determine which block in the sequence's list of blocks holds the token.
    block_list_idx = token_idx // self.block_size
    
    # Determine the token's offset within that block.
    offset_in_block = token_idx % self.block_size
    
    # Get the physical block from the sequence's block table.
    physical_block = self.block_tables[seq_id][block_list_idx]
    
    # Return the physical block number and the offset.
    return (physical_block.block_num, offset_in_block)
    ### END CODE HERE ###

# Monkey-patch the method into our class for testing
SimulatedKVCacheManager.get_physical_location = get_physical_location

In [ ]:
# Test cell for Exercise 3 and Final Integration

print("--- Testing Exercise 3 and Full Integration ---")

manager = SimulatedKVCacheManager(block_size=4, num_blocks=10)

# The free blocks are [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].
# They will be allocated in reverse order: 9, 8, 7, ...

# Scenario:
# - Sequence 301 will have 9 tokens. It should use 3 blocks.
#   - Tokens 0-3 in Block 9
#   - Tokens 4-7 in Block 8
#   - Token 8 in Block 7
# - Sequence 302 will have 3 tokens. It should use 1 block.
#   - Tokens 0-2 in Block 6

print("\nSimulating sequence generation...")
manager.allocate_sequence(301)
for _ in range(8): # Append 8 more tokens to reach a total length of 9
    manager.append_token(301)

manager.allocate_sequence(302)
for _ in range(2): # Append 2 more tokens to reach a total length of 3
    manager.append_token(302)
print("Simulation complete.")

print("\nChecking physical memory mapping for Sequence 301 (length 9)...")
# First token of the sequence
loc1 = manager.get_physical_location(301, 0)
assert loc1 == (9, 0), f"Test 1 Failed: Got {loc1}, expected (9, 0)"
print(f"Token 0 -> {loc1} ✅")

# Last token of the first block
loc2 = manager.get_physical_location(301, 3)
assert loc2 == (9, 3), f"Test 2 Failed: Got {loc2}, expected (9, 3)"
print(f"Token 3 -> {loc2} ✅")

# First token of the second block
loc3 = manager.get_physical_location(3